# Discover sentiment vocabulary with “and”

Load all reviews in `data/reviews`, including reviews without labels. Load the seed lists directly from `SemanticAnalyzer.POSITIVE_WORDS` and `NEGATIVE_WORDS`.

For each sentence, match adjacent tokens `seed and candidate` or `candidate and seed`. The candidate must not already be in either seed list and must not be a stop word or negation. Keep all tokens while matching so removing stop words cannot create false adjacency. Sentence and contrast-clause boundaries are preserved; punctuation within a clause is discarded by the tokenizer.

Count each matching pair occurrence, separately for positive and negative seeds. Also report distinct reviews containing the candidate, matched seeds, and up to three examples. Duplicate review IDs within each game are counted once. No minimum frequency is imposed by default.

These are vocabulary **candidates**, not confirmed sentiment words: conjunctions can connect nouns and unrelated qualities. Negation and sarcasm are not interpreted. Inspect mixed associations and examples before editing the seed lists; this notebook does not modify them.


In [25]:
import csv
import json
import sys
from collections import Counter
from pathlib import Path

PROJECT_ROOT = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "src").is_dir() and (path / "notebooks").is_dir()), None,
)
if PROJECT_ROOT is None:
    raise RuntimeError("Start from the project root or notebooks folder")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.csv_io import REVIEWS_DATA_DIR, load_reviews
from src.review_parser import ReviewParser
from src.semantic_analyzer import SemanticAnalyzer

parser = ReviewParser()
positive_words = {word.lower() for word in SemanticAnalyzer.POSITIVE_WORDS}
negative_words = {word.lower() for word in SemanticAnalyzer.NEGATIVE_WORDS}
known_words = positive_words | negative_words
excluded_words = parser.stop_words | {"and", "not", "no", "never", "neither", "nor"}
MIN_ASSOCIATIONS = 1
OUTPUT_FILE = PROJECT_ROOT / "data" / "potential_semantic_words_but.csv"
print(f"Loaded {len(positive_words)} positive and {len(negative_words)} negative seeds")


Loaded 20 positive and 19 negative seeds


In [26]:
def find_associations(text):
    """Yield (candidate, seed, polarity, sentence) for each adjacent pair."""
    sentences = parser.split_sentences(text)
    for tokens in parser.tokenize(sentences):
        for index in range(1, len(tokens) - 1):
            if tokens[index] != "but":
                continue
            left, right = tokens[index - 1], tokens[index + 1]
            for seed, candidate in ((left, right), (right, left)):
                if (candidate in known_words or candidate in excluded_words
                        or not candidate.isalpha() or seed not in known_words):
                    continue
                if seed in positive_words:
                    yield candidate, seed, "positive", tokens
                if seed in negative_words:
                    yield candidate, seed, "negative", tokens


In [27]:
candidates = {}
review_count = duplicate_rows = matching_pairs = 0
files = sorted(
    (path for path in REVIEWS_DATA_DIR.glob("*.csv")
     if path.stem.isascii() and path.stem.isdigit()
     and str(int(path.stem)) == path.stem and 0 < int(path.stem) <= 2**32 - 1),
    key=lambda path: int(path.stem),
)
for index, path in enumerate(files):
    unique_reviews = {}
    for review in load_reviews(int(path.stem)):
        review_count += 1
        seen_candidates = set()
        for candidate, seed, polarity, sentence in find_associations(review.review):
            entry = candidates.setdefault(candidate, {
                "positive_count": 0, "negative_count": 0, "review_count": 0,
                "positive_seeds": Counter(), "negative_seeds": Counter(),
            })
            entry[f"{polarity}_count"] += 1
            entry[f"{polarity}_seeds"][seed] += 1
            matching_pairs += 1
            if candidate not in seen_candidates:
                entry["review_count"] += 1
                seen_candidates.add(candidate)
    if index % 100 == 0 or index == len(files):
        print(f"Processed {index}/{len(files)} game files", flush=True)

summary = []
for word, entry in candidates.items():
    positive, negative = entry["positive_count"], entry["negative_count"]
    total = positive + negative
    if total < MIN_ASSOCIATIONS:
        continue
    summary.append({
        "word": word, "positive_count": positive, "negative_count": negative,
        "total_associations": total, "review_count": entry["review_count"],
        "positive_share": positive / total,
        "suggested_polarity": "positive" if positive > negative else "negative" if negative > positive else "mixed",
        "positive_seeds": dict(entry["positive_seeds"]),
        "negative_seeds": dict(entry["negative_seeds"]),
    })
summary.sort(key=lambda row: (-row["total_associations"], row["word"]))
print(f"Reviewed {review_count} unique reviews; excluded {duplicate_rows} duplicate rows")
print(f"Found {matching_pairs} pairs and {len(summary)} candidate words")
print(f"{'Word':24} {'Positive':>9} {'Negative':>9} {'Reviews':>9} {'Suggested':>10}")
for row in summary[:50]:
    print(f"{row['word']:24} {row['positive_count']:9} {row['negative_count']:9} "
          f"{row['review_count']:9} {row['suggested_polarity']:>10}")


Processed 0/969 game files
Processed 100/969 game files
Processed 200/969 game files
Processed 300/969 game files
Processed 400/969 game files
Processed 500/969 game files
Processed 600/969 game files
Processed 700/969 game files
Processed 800/969 game files
Processed 900/969 game files
Reviewed 92506 unique reviews; excluded 0 duplicate rows
Found 2802 pairs and 878 candidate words
Word                      Positive  Negative   Reviews  Suggested
there                           85        24       109   positive
simple                          97         3        95   positive
if                              62        20        82   positive
also                            64        17        81   positive
nothing                         55        11        65   positive
after                           48         9        57   positive
short                           56         1        56   positive
still                           24        28        52   negative
when                

In [28]:
OUTPUT_FILE.parent.mkdir(parents=True, exist_ok=True)
fields = ["word", "positive_count", "negative_count", "total_associations", "review_count",
          "positive_share", "suggested_polarity", "positive_seeds", "negative_seeds"]
with OUTPUT_FILE.open("w", encoding="utf-8", newline="") as file:
    writer = csv.DictWriter(file, fieldnames=fields)
    writer.writeheader()
    for row in summary:
        writer.writerow({key: json.dumps(value, ensure_ascii=False) if isinstance(value, (dict, list))
                         else value for key, value in row.items()})
print(f"Saved candidate summary to {OUTPUT_FILE}")

Saved candidate summary to C:\projects\GameRecommender\data\potential_semantic_words_but.csv
